# Color Picker Experiment

Autonomous color mixing via Bayesian optimization, running cell-by-cell using `ColorPickerNotebook`.

## Setup
- Set `LAB_SERVER_URL` below (or configure via `experiment.settings.yaml` / `EXPERIMENT_LAB_SERVER_URL` env var).
- Each iteration pipettes a batch of colors onto the plate, images it, and feeds the measured RGB values back into the Bayesian solver.
- Run cells sequentially. Re-run individual iteration cells to repeat a step without restarting the experiment.

## Configuration & Setup

In [ ]:
from colorpicker_experiment import ColorPickerNotebook

# ── Lab connection ────────────────────────────────────────────────────────────
LAB_SERVER_URL = "http://parker.cels.anl.gov:8000"  # adjust as needed

# ── Experiment parameters (override config file defaults) ─────────────────────
OPENTRON = "ot2_gamma"  # OT-2 node name
PIPETTE_SIDE = "left"  # 'left' or 'right'
ITERATIONS = 4  # number of Bayesian optimization rounds
POP_SIZE = 4  # colors mixed per iteration

exp = ColorPickerNotebook(
    lab_server_url=LAB_SERVER_URL,
    opentron=OPENTRON,
    pipette_side=PIPETTE_SIDE,
    iterations=ITERATIONS,
    pop_size=POP_SIZE,
)

exp.logger.info(
    "Experiment configured",
    experiment_name=exp.experiment_design.experiment_name,
    workcell_url=getattr(exp, "workcell_server_url", "auto-discover from lab"),
)

## Start Experiment & Initialize Run

In [ ]:
exp.start(run_name="Color Picker notebook run")
exp._initialize_run(opentron=OPENTRON, pipette_side=PIPETTE_SIDE)

exp.display(
    {"target_color": exp.target_color, "iterations": ITERATIONS, "pop_size": POP_SIZE},
    title="Run Parameters",
)

## Iterations Loop

In [ ]:
for i in ITERATIONS:
    colors = exp.loop(i)
    exp.display(colors, title=f"Iteration {i} — measured well colors")

## Results & End Experiment

In [ ]:
import numpy as np

best_idx = int(
    np.argmin(exp.solver._grade_population(exp.previous_colors, exp.target_color))
)

results = {
    "target_color": exp.target_color,
    "best_color": exp.previous_colors[best_idx],
    "total_wells_used": exp.total_wells,
    "iterations_completed": ITERATIONS,
}

exp.display(results, title="Experiment Results")
exp.end()